# EP12 — Decision Tables & Knowledge Engineering
**COMPSCI 713 · S1 2025 Q3 · 2 marks**

**Question:** What are two advantages of Decision Tables in the Knowledge Engineering process?

---

## Background: Knowledge Engineering (KE) vs Machine Learning

**Knowledge Engineering:**
- Experts are *interviewed* to extract the rules they apply
- Rules are explicitly encoded into a system
- No training data required — knowledge is curated
- Examples: expert systems, medical diagnosis, legal reasoning

**Machine Learning:**
- Rules are *learned* from data automatically
- No expert interview needed, but large labelled datasets required

**Decision Tables** are one of the most commonly used tools in KE to represent expert rules in a structured, readable format.

---

## What is a Decision Table?

A decision table maps combinations of **conditions** (input values) to **actions** (decisions/outputs).

| Example | Loan Approval Decision Table |
|---------|------|

| Credit Score | Income Level | Debt Ratio | Decision |
|-------------|-------------|------------|----------|
| High | High | Low | Approve |
| High | Low | Low | Approve |
| High | Any | High | Review |
| Low | High | Low | Review |
| Low | Low | Any | Reject |
| Low | Any | High | Reject |

Each column = a condition or action. Each row = one rule.

---

## Two Advantages of Decision Tables in KE

### Advantage 1 — Easy to validate with domain experts
Decision tables present rules in a tabular, human-readable format that domain experts (who are not programmers) can directly read and verify.
- The expert can scan rows and immediately spot errors: "That's wrong — a high credit score with low debt should always be approved, not reviewed."
- This closes the **knowledge acquisition gap**: the expert's mental model can be directly checked against what the system will do.

### Advantage 2 — Automatic completeness and consistency checking
A decision table's structure allows tools (or even manual inspection) to check whether:
- **Completeness:** every possible combination of conditions has a corresponding rule (no gaps)
- **Consistency:** no two rules specify the same conditions but different actions (no contradictions)
- **Redundancy:** no rules are duplicates

In free-form interviews or prose rules, these gaps and contradictions are often invisible. A table makes them immediately apparent.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from itertools import product

# =============================================================
# Build a Decision Table and check completeness + consistency
# Domain: Loan approval expert system
# =============================================================

# Conditions with their possible values
conditions = {
    'Credit Score': ['High', 'Low'],
    'Income Level': ['High', 'Low'],
    'Debt Ratio':   ['Low',  'High'],
}

# Expert rules (from interview)
# Each rule: (Credit, Income, Debt) -> Decision
expert_rules = [
    ('High', 'High', 'Low',  'Approve'),
    ('High', 'Low',  'Low',  'Approve'),
    ('High', 'High', 'High', 'Review'),
    ('High', 'Low',  'High', 'Review'),
    ('Low',  'High', 'Low',  'Review'),
    # Note: ('Low', 'Low', 'Low', ?) is MISSING — completeness gap!
    ('Low',  'High', 'High', 'Reject'),
    ('Low',  'Low',  'High', 'Reject'),
]

# All possible combinations
all_combinations = list(product(*conditions.values()))
combination_keys  = list(conditions.keys())

print('=== Decision Table ===')
df = pd.DataFrame(expert_rules, columns=['Credit', 'Income', 'Debt', 'Decision'])
print(df.to_string(index=False))

# --- Completeness check ---
covered = {tuple(r[:3]) for r in expert_rules}
missing = [c for c in all_combinations if c not in covered]

print(f'\n--- Completeness Check ---')
print(f'Total possible combinations: {len(all_combinations)}')
print(f'Rules defined:               {len(expert_rules)}')
if missing:
    print(f'⚠️  MISSING combinations ({len(missing)}):')
    for m in missing: print(f'   {dict(zip(combination_keys, m))} → ??? (no rule defined!)')
else:
    print('✅ Complete — all combinations covered')

# --- Consistency check (duplicate condition → different action) ---
print(f'\n--- Consistency Check ---')
condition_to_actions = {}
for r in expert_rules:
    key = tuple(r[:3])
    condition_to_actions.setdefault(key, set()).add(r[3])

inconsistent = {k: v for k, v in condition_to_actions.items() if len(v) > 1}
if inconsistent:
    print('⚠️  INCONSISTENCIES found:')
    for combo, actions in inconsistent.items():
        print(f'   {dict(zip(combination_keys, combo))} → {actions}')
else:
    print('✅ Consistent — no contradictions')

In [ ]:
# =============================================================
# Visualisation: Decision table as a heatmap
# Shows which condition combinations are covered
# =============================================================
decision_map = {'Approve': 2, 'Review': 1, 'Reject': 0, None: -1}
colour_map   = {2: '#2ecc71', 1: '#f39c12', 0: '#e74c3c', -1: '#ecf0f1'}
labels_map   = {2: 'Approve', 1: 'Review', 0: 'Reject', -1: 'NOT\nDEFINED'}

# Build grid: rows = (Credit, Income), cols = Debt
combos_2d = list(product(['High', 'Low'], ['High', 'Low']))  # (Credit, Income)
debt_vals  = ['Low', 'High']

grid = np.zeros((4, 2), dtype=int)
grid_labels = [[''] * 2 for _ in range(4)]
for i, (credit, income) in enumerate(combos_2d):
    for j, debt in enumerate(debt_vals):
        key = (credit, income, debt)
        match = [r for r in expert_rules if tuple(r[:3]) == key]
        if match:
            grid[i][j] = decision_map[match[0][3]]
            grid_labels[i][j] = match[0][3]
        else:
            grid[i][j] = -1
            grid_labels[i][j] = 'MISSING'

fig, ax = plt.subplots(figsize=(7, 5))
for i in range(4):
    for j in range(2):
        val = grid[i][j]
        ax.add_patch(plt.Rectangle((j, 3-i), 1, 1,
                                    color=colour_map[val], ec='white', lw=2))
        ax.text(j+0.5, 3-i+0.5, grid_labels[i][j],
                ha='center', va='center', fontsize=9, fontweight='bold',
                color='white' if val != -1 else '#c0392b')

row_labels = [f'Credit:{c} / Income:{inc}' for c, inc in combos_2d]
for i, label in enumerate(reversed(row_labels)):
    ax.text(-0.05, i+0.5, label, ha='right', va='center', fontsize=8)

for j, d in enumerate(debt_vals):
    ax.text(j+0.5, 4.15, f'Debt: {d}', ha='center', va='bottom', fontsize=9, fontweight='bold')

for colour, label in [('#2ecc71','Approve'), ('#f39c12','Review'),
                       ('#e74c3c','Reject'),  ('#ecf0f1','MISSING')]:
    ax.add_patch(plt.Rectangle((0,0), 0, 0, color=colour, label=label))

ax.set_xlim(-1.8, 2.2)
ax.set_ylim(-0.2, 4.5)
ax.axis('off')
ax.set_title('Decision Table Visualisation\n'
             'Red text = completeness gap (expert never specified this case)', fontsize=11)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

print('\nKey insight: The MISSING cell is immediately visible in the table.')
print('In a free-text interview, this gap would likely go unnoticed.')

In [ ]:
# =============================================================
# Bonus: compare Decision Table vs free-text rules
# Shows the validation advantage concretely
# =============================================================

free_text_rules = [
    "If credit score is high and debt is low, approve the loan.",
    "If credit score is high and debt is high, send for review.",
    "If income is high and credit is low and debt is low, send for review.",
    "If debt is high and credit is low, reject the loan.",
    # Note: Low credit, low income, low debt — quietly missing
]

print('=== Free-text Rules (from expert interview) ===')
for i, rule in enumerate(free_text_rules, 1):
    print(f'  Rule {i}: {rule}')

print('\nCan you spot the gap? → Low credit, Low income, Low debt is never covered.')
print('In a table: the empty cell is visually obvious.')
print('In prose: the missing case is invisible until a borrower hits it in production.')
print()
print('This is the core advantage: Decision Tables make')
print('completeness gaps and contradictions STRUCTURALLY VISIBLE.')

## Exam Quick-Reference

**Two advantages of Decision Tables in KE:**

1. **Transparent to domain experts** — the table format is directly readable by non-programmers. Experts can verify rules row by row, catching errors that would be hidden in code or prose.

2. **Enables completeness and consistency checking** — the table structure makes it possible to systematically check whether all condition combinations have a rule (completeness) and whether any two rules contradict each other (consistency). These checks are difficult or impossible with free-text rules.

**Be concise:** one sentence each is enough for 1 mark per advantage.